In [17]:
import os
import math
import glob
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from torch.optim import Adam
from torchinfo import summary
from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import PeakSignalNoiseRatio
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
HR_train_paths = sorted(glob.glob("./data/DIV2K_train_HR/*.png"))
X2_train_paths = sorted(glob.glob("./data/DIV2K_train_LR_bicubic/X2/*.png"))
X4_train_paths = sorted(glob.glob("./data/DIV2K_train_LR_bicubic/X4/*.png"))
X8_train_paths = sorted(glob.glob("./data/DIV2K_train_LR_bicubic/X8/*.png"))
X16_train_paths = sorted(glob.glob("./data/DIV2K_train_LR_bicubic/X16/*.png"))
X32_train_paths = sorted(glob.glob("./data/DIV2K_train_LR_bicubic/X32/*.png"))
X64_train_paths = sorted(glob.glob("./data/DIV2K_train_LR_bicubic/X64/*.png"))

HR_valid_paths = sorted(glob.glob("./data/DIV2K_valid_HR/*.png"))
X2_valid_paths = sorted(glob.glob("./data/DIV2K_valid_LR_bicubic/X2/*.png"))
X4_valid_paths = sorted(glob.glob("./data/DIV2K_valid_LR_bicubic/X4/*.png"))
X8_valid_paths = sorted(glob.glob("./data/DIV2K_valid_LR_bicubic/X8/*.png"))
X16_valid_paths = sorted(glob.glob("./data/DIV2K_valid_LR_bicubic/X16/*.png"))
X32_valid_paths = sorted(glob.glob("./data/DIV2K_valid_LR_bicubic/X32/*.png"))
X64_valid_paths = sorted(glob.glob("./data/DIV2K_valid_LR_bicubic/X64/*.png"))

## Model and data preparation

First let's calculate the mean RGB value of the DIV2K dataset, it will be subtracted from the input during training and so it has to be during inference.

In [25]:
# R, G, B = 0, 0, 0
# total_pixels = 0
# for path in tqdm(HR_train_paths):
#     img = Image.open(path).convert('RGB')
#     img_np = np.array(img) / 255.0

#     h, w, _ = img_np.shape
#     total_pixels += h * w
    
#     R += np.sum(img_np[:, :, 0])
#     G += np.sum(img_np[:, :, 1])
#     B += np.sum(img_np[:, :, 2])

# R /= total_pixels
# G /= total_pixels
# B /= total_pixels
# R, G, B

  0%|          | 0/800 [00:00<?, ?it/s]

(np.float64(0.44882884613943946),
 np.float64(0.43713809810624193),
 np.float64(0.4040371984052683))

## SRResNet

The architecture below is described in this [paper](https://arxiv.org/pdf/1609.04802)

Although I am **not going to train SRResNet**, _EDSR_ is a slight modification of it and it is impossible to build _EDSR_ without understanding _SRResNet_.

In [8]:
class ResBlockSRRN(nn.Module):
        def __init__(self):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv2d(64, 64, 3, stride=1, padding='same'),
                nn.BatchNorm2d(64),
                nn.PReLU(),
                nn.Conv2d(64, 64, 3, stride=1, padding='same'),
                nn.BatchNorm2d(64)
            )

        def forward(self, x):
            return x + self.block(x)

class SRResNet(nn.Module):
    def __init__(self, n: int):
        """
        Args:
            n: scaling factor
        """
        super().__init__()
        
        self.expand = nn.Sequential(
            nn.Conv2d(3, 64, 9, stride=1, padding='same'),
            nn.PReLU()
        )

        self.residual_blocks = nn.Sequential()
        for _ in range(16):
            self.residual_blocks.append(ResBlockSRRN())

        self.residual_blocks.append(nn.Conv2d(64, 64, 3, stride=1, padding='same'))
        self.residual_blocks.append(nn.BatchNorm2d(64))

        self.upscaling_head = nn.Sequential()
        for _ in range(int(math.log2(n))):
            self.upscaling_head.append(nn.Conv2d(64, 256, 3, stride=1, padding='same'))
            self.upscaling_head.append(nn.PixelShuffle(2))
            self.upscaling_head.append(nn.PReLU())
            
        self.upscaling_head.append(nn.Conv2d(64, 3, 9, stride=1, padding='same'))

    def forward(self, x):
        x = self.expand(x)
        return self.upscaling_head(self.residual_blocks(x) + x)

In [9]:
summary(SRResNet(4), input_size=(16, 3, 48, 48))

Layer (type:depth-idx)                   Output Shape              Param #
SRResNet                                 [16, 3, 192, 192]         --
├─Sequential: 1-1                        [16, 64, 48, 48]          --
│    └─Conv2d: 2-1                       [16, 64, 48, 48]          15,616
│    └─PReLU: 2-2                        [16, 64, 48, 48]          1
├─Sequential: 1-2                        [16, 64, 48, 48]          --
│    └─ResBlockSRRN: 2-3                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-1              [16, 64, 48, 48]          74,113
│    └─ResBlockSRRN: 2-4                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-2              [16, 64, 48, 48]          74,113
│    └─ResBlockSRRN: 2-5                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-3              [16, 64, 48, 48]          74,113
│    └─ResBlockSRRN: 2-6                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-4              [16, 64, 48, 48]          74,

## EDSR - Enhanced Deep Super Resolution

This architecture is taken from this [paper](https://arxiv.org/pdf/1707.02921#page=9&zoom=100,66,222)

Below, aside from the reproduction of EDSR there is also EDSRLight in case my GPU won't be able to handle EDSR in a reasonable ammount of time. Light version has fewer residual blocks, smaller feature dimension, and doesn't use constant scaling in residual block.

In [10]:
class ResBlockEDSRLight(nn.Module):
        def __init__(self):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv2d(64, 64, 3, stride=1, padding='same'),
                nn.PReLU(),
                nn.Conv2d(64, 64, 3, stride=1, padding='same'),
            )

        def forward(self, x):
            return x + self.block(x)

class EDSRLight(nn.Module):
    def __init__(self, n: int):
        """
        Args:
            n: scaling factor
        """
        super().__init__()
        self.expand = nn.Sequential(
            nn.Conv2d(3, 64, 9, stride=1, padding='same'),
            nn.PReLU()
        )

        self.residual_blocks = nn.Sequential()
        for _ in range(16):
            self.residual_blocks.append(ResBlockEDSRLight())

        self.residual_blocks.append(nn.Conv2d(64, 64, 3, stride=1, padding='same'))

        self.upscaling_head = nn.Sequential()
        for _ in range(int(math.log2(n))):
            self.upscaling_head.append(nn.Conv2d(64, 4*64, 3, stride=1, padding='same'))
            self.upscaling_head.append(nn.PixelShuffle(2))
            self.upscaling_head.append(nn.PReLU())
            
        self.upscaling_head.append(nn.Conv2d(64, 3, 9, stride=1, padding='same'))

    def forward(self, x):
        x = self.expand(x)
        return self.upscaling_head(self.residual_blocks(x) + x)

class ResBlockEDSR(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(256, 256, 3, stride=1, padding='same'),
            nn.PReLU(),
            nn.Conv2d(256, 256, 3, stride=1, padding='same'),
        )

    def forward(self, x):
        return x + 0.1*self.block(x)

class EDSR(nn.Module):
    def __init__(self, n: int):
        """
        Args:
            n: scaling factor
        """
        super().__init__()
        self.expand = nn.Sequential(
            nn.Conv2d(3, 256, 9, stride=1, padding='same'),
            nn.PReLU()
        )

        self.residual_blocks = nn.Sequential()
        for _ in range(32):
            self.residual_blocks.append(ResBlockEDSR())

        self.residual_blocks.append(nn.Conv2d(256, 256, 3, stride=1, padding='same'))

        self.upscaling_head = nn.Sequential()
        for _ in range(int(math.log2(n))):
            self.upscaling_head.append(nn.Conv2d(256, 4*256, 3, stride=1, padding='same'))
            self.upscaling_head.append(nn.PixelShuffle(2))
            self.upscaling_head.append(nn.PReLU())
            
        self.upscaling_head.append(nn.Conv2d(256, 3, 9, stride=1, padding='same'))

    def forward(self, x):
        x = self.expand(x)
        return self.upscaling_head(self.residual_blocks(x) + x)

In [13]:
summary(EDSR(16, 64, 4), input_size=(16, 3, 48, 48))

Layer (type:depth-idx)                   Output Shape              Param #
ESDR                                     [16, 3, 192, 192]         --
├─Sequential: 1-1                        [16, 64, 48, 48]          --
│    └─Conv2d: 2-1                       [16, 64, 48, 48]          15,616
│    └─PReLU: 2-2                        [16, 64, 48, 48]          1
├─Sequential: 1-2                        [16, 64, 48, 48]          --
│    └─ResBlockESDR: 2-3                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-1              [16, 64, 48, 48]          73,857
│    └─ResBlockESDR: 2-4                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-2              [16, 64, 48, 48]          73,857
│    └─ResBlockESDR: 2-5                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-3              [16, 64, 48, 48]          73,857
│    └─ResBlockESDR: 2-6                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-4              [16, 64, 48, 48]          73,

The dataset is mostly the same as for _FSRCNN_, but it also includes data augmentation with **horizontal flips** and **90 degree rotations**. It also subtracts average values of RGB in DIV2K dataset.

In [ ]:
class EDSR_Dataset(Dataset):
    def __init__(self, target_paths: list[str], scale: int, ram_limit_gb: float = 2.0):
        self.crop_size = scale * 48
        self.scale = scale
        self.DIV2K_RGB = torch.tensor([0.44882884613943946, 0.43713809810624193, 0.4040371984052683], device=device)

        self.rotations = [0, 90, 180, 270]
        self.transforms = v2.Compose([
            v2.PILToTensor(),
            v2.Lambda(lambda x: (x / 255.0) - self.DIV2K_RGB)
        ])

        self.preloaded = {}
        self.paths = target_paths

        total_ram_used = 0
        for i, path in enumerate(tqdm(target_paths, desc="Preloading images")):
            img = Image.open(path).convert("RGB")
            total_ram_used += img.width * img.height * 3 / (1024 ** 3)  # ~size in GB

            if total_ram_used < ram_limit_gb:
                self.preloaded[i] = img
            else:
                break

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        if idx in self.preloaded:
            target = self.preloaded[idx]
        else:
            target = Image.open(self.paths[idx]).convert("RGB")

        target = self.random_crop(target, self.crop_size)
        inp = target.resize((target.width // self.scale, target.height // self.scale), Image.BICUBIC)
        
        rotation =  random.choice(self.rotations)
        if rotation != 0:
            inp = v2.functional.rotate(inp, rotation)
            target = v2.functional.rotate(target, rotation)
        if random.randint(0, 1):
            inp = v2.functional.horizontal_flip(inp)
            target = v2.functional.horizontal_flip(target)
            
        return self.transforms(inp), self.transforms(target)

    def random_crop(self, img, size):
        w, h = img.size
        if w < size or h < size:
            img = img.resize((size, size), Image.BICUBIC)
        x = random.randint(0, w - size)
        y = random.randint(0, h - size)
        return img.crop((x, y, x + size, y + size))

    def set_scale(self, scale: int):
        self.scale = scale

    def set_crop_size(self, crop_size: int):
        self.crop_size = crop_size

In [ ]:
psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
lpips = LearnedPerceptualImagePatchSimilarity(normalize=True).to(device)

In [ ]:
DIV2K_RGB = torch.tensor([0.44882884613943946, 0.43713809810624193, 0.4040371984052683], device=device)
transform = v2.Compose([
    v2.PILToTensor(),
    # subtract DIV2K mean RGB, as in training
    v2.Lambda(lambda x: (x / 255.0) - DIV2K_RGB)
])

Metric calculation stays mostly the same as in FSRCNN, but to ensure that subtraction of `DIV2K_RGB` doesn't cause any harm, I am adding it back here, I will do the same when calculating metrics during training.

In [ ]:
def calc_metrics(model: FSRCNN, target_ds: list[str], scale: int):
    transform_target = v2.Compose([
        v2.PILToTensor(),
        v2.Lambda(lambda x: x/255.0)
    ])

    transform_input = v2.Compose([
        v2.PILToTensor(),
        v2.Lambda(lambda x: (x / 255.0) - DIV2K_RGB)
    ])

    psnr_acc = 0
    ssim_acc = 0
    lpips_acc = 0
    failed_lpips = 0

    model.eval()
    for i in tqdm(range(len(target_ds)), leave=False):
        target_image = Image.open(target_ds[i]).convert("RGB")
        w, h = target_image.size

        w -= w % scale
        h -= h % scale
        target_image = target_image.crop((0, 0, w, h))
        
        lowres = target_image.resize((w // scale, h // scale), resample=Image.BICUBIC)
        input_tensor = transform_input(lowres).unsqueeze(0).to(device)
        target_tensor = transform_target(target_image).unsqueeze(0).to(device)

        with torch.inference_mode():
            sr = (model(input_tensor) + DIV2K_RGB).clamp(0, 1)

        psnr_acc += psnr(sr, target_tensor).item()
        ssim_acc += ssim(sr, target_tensor).item()
        
        # There are 2 images that cause lpips to fail
        try:
            x = lpips(sr, target_tensor).cpu().item()
            if np.isnan(x):
                failed_lpips += 1
                continue
                
            lpips_acc += x
        except:
            failed_lpips += 1

    lpips_acc /= len(target_ds) - failed_lpips
    psnr_acc /= len(target_ds)
    ssim_acc /= len(target_ds)
    return psnr_acc, ssim_acc, lpips_acc

In [ ]:
def train_step(model, dataloader, optimizer, loss_fn):
    avg_psnr = 0
    avg_ssim = 0
    model.train()

    for batch, target in dataloader:
        batch, target = batch.to(device), target.to(device)
        
        logits = model(batch)
        loss = loss_fn(logits, target)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        logits += DIV2K_RGB
        target += DIV2K_RGB

        logits = logits.clamp(0.0, 1.0)
        target = target.clamp(0.0, 1.0)
        
        avg_psnr += psnr(logits, target).item()
        avg_ssim += ssim(logits, target).item()
        
    avg_psnr /= len(dataloader)
    avg_ssim /= len(dataloader)
    return avg_psnr, avg_ssim

def valid_step(model, dataloader, loss_fn):
    avg_psnr = 0
    avg_ssim = 0
    avg_lpips = 0
    model.eval()

    with torch.inference_mode():
        for batch, target in dataloader:
            batch, target = batch.to(device), target.to(device)
            
            logits = model(batch) + DIV2K_RGB
            target += DIV2K_RGB
            
            logits = logits.clamp(0.0, 1.0)
            target = target.clamp(0.0, 1.0)
            
            avg_psnr += psnr(logits, target).item()
            avg_ssim += ssim(logits, target).item()
            avg_lpips += lpips(logits, target).item()

    avg_psnr /= len(dataloader)
    avg_ssim /= len(dataloader)
    avg_lpips /= len(dataloader)

        
    return avg_psnr, avg_ssim, avg_lpips

In [ ]:
def train(model, train_dl, valid_dl, optimizer, scheduler: ReduceLROnPlateau, loss_fn, epochs, start_checkpoint=None):
    os.makedirs('./tmp_model_checkpoints', exist_ok=True)
    counter = 0 # count epochs without printing training stats
    log_freq = epochs // 100 # how often to print stats when no progress is made
    
    if start_checkpoint:
        start_epoch = start_checkpoint['epoch']
        best_psnr = start_checkpoint['best_psnr']
        best_ssim = start_checkpoint['best_ssim']
        best_lpips = start_checkpoint['best_lpips']
    else:
        start_epoch = 0
        best_psnr = 0
        best_ssim = 0
        best_lpips = float('inf')
        
    for epoch in tqdm(range(start_epoch, epochs), desc="Epochs"):
        counter += 1
        train_psnr, train_ssim = train_step(
            model,
            train_dl,
            optimizer,
            loss_fn
        )

        valid_psnr, valid_ssim, valid_lpips = valid_step(
            model,
            valid_dl,
            loss_fn,
        )

        scheduler.step(valid_psnr)

        progress = False
        
        if valid_psnr > best_psnr:
            progress = True
            best_psnr = valid_psnr
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }
            torch.save(checkpoint, f'./tmp_model_checkpoints/best_psnr.pth')

        if valid_ssim > best_ssim:
            progress = True
            best_ssim = valid_ssim
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }
            torch.save(checkpoint, f'./tmp_model_checkpoints/best_ssim.pth')

        if valid_lpips < best_lpips:
            progress = True
            best_lpips = valid_lpips
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }
            torch.save(checkpoint, f'./tmp_model_checkpoints/best_lpips.pth')

        if epoch == epochs-1:
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }
            torch.save(checkpoint, f'./tmp_model_checkpoints/last.pth')
            
        if progress or counter >= log_freq:
            counter = 0
            print(
                f"Epoch: {epoch+1} | "
                f"learning rate: {scheduler.get_last_lr()[0]:.7f} | "
                f"[train] PSNR: {train_psnr:.4f} | "
                f"[train] SSIM: {train_ssim:.4f} | "
                f"[valid] PSNR: {valid_psnr:.4f} | "
                f"[valid] SSIM: {valid_ssim:.4f} | "
                f"[valid] LPIPS: {valid_lpips:.4f}"
            )

## X2 Scaling

### Training

In [15]:
valid_ds = EDSR_Dataset(HR_valid_paths, 2, ram_limit_gb=1)

In [ ]:
train_ds = EDSR_Dataset(HR_train_paths, 2, ram_limit_gb=8)

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

model = EDSR(2).to(device)
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=0.003)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=10, factor=0.8, min_lr=1e-7) # don't forget to update this

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 20000)

### Geometric Self-ensemble

### Super-resolution showcase